# Predictive Maintenance mit dem Scania-APS-Datensatz

Dieses Notebook zeigt einen kompakten CRISP-DM-Ablauf fuer eine unausgewogene Klassifikationsaufgabe: Daten pruefen, eine einfache Baseline aufstellen, fehlende Werte in einer Pipeline behandeln und Fehlentscheidungen kostenbewusst auswerten.

**Quelle:** [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/421/aps+failure+at+scania+trucks). Das Archiv umfasst 60.000 Trainings- und 16.000 Testfaelle. Die Trainingsdaten enthalten 1.000 positive Faelle und 170 anonymisierte Betriebsmerkmale.

**Wichtige Grenze:** Die Daten sind einzelne, nicht zeitlich geordnete Beobachtungen. Das Modell sagt weder den Zeitpunkt noch die verbleibende Zeit bis zu einem Fehler voraus. Die anonymisierten Merkmale erlauben ausserdem nur eine eingeschraenkte technische Interpretation.

Die UCI-Herausforderung setzt Kosten von 10 fuer einen Fehlalarm (FP) und 500 fuer einen uebersehenen APS-Fehler (FN) an. Diese Werte sind eine Benchmark-Annahme und muessen in einer realen Anwendung durch betriebliche Kosten ersetzt werden.

## 1. Daten laden und einen ersten Ueberblick gewinnen

Der Download enthaelt beide offiziellen Dateien. Fuer diese Uebung verwenden wir nur den Trainingsdatensatz und teilen ihn anschliessend stratifiziert in Training, Validierung und Test auf.

In [ ]:
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (average_precision_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

In [ ]:
archive_url = "https://archive.ics.uci.edu/static/public/421/aps+failure+at+scania+trucks.zip"
with urlopen(archive_url) as response:
    archive_bytes = response.read()

with ZipFile(BytesIO(archive_bytes)) as archive:
    training_file = next(
        name for name in archive.namelist()
        if name.endswith("aps_failure_training_set.csv")
    )
    with archive.open(training_file) as csv_file:
        df = pd.read_csv(csv_file, na_values=["na", "NA"])

df.shape

In [ ]:
display(df.head())
display(df["class"].value_counts().rename("Anzahl"))
missing_rate = df.drop(columns="class").isna().mean().sort_values(ascending=False)
display(missing_rate.head(10).rename("Fehlender Anteil"))

df["class"].value_counts().plot.bar(color=["steelblue", "tomato"])
plt.title("Zielklassen im Trainingsdatensatz")
plt.ylabel("Anzahl der Faelle")
plt.xticks(rotation=0)
plt.show()

Ein Modell, das immer die Mehrheitsklasse `neg` vorhersagt, erreicht bereits etwa 98,3 % Accuracy. Dieser Wert ist keine ausreichende Erfolgsmessung, weil damit jeder positive APS-Fehler uebersehen wird. Wir halten die seltene positive Klasse durch stratifizierte Aufteilungen in allen Teilmengen vertreten.

In [ ]:
X = df.drop(columns="class")
y = df["class"].map({"neg": 0, "pos": 1})
if y.isna().any():
    raise ValueError("Unerwartete Werte in der Zielspalte 'class'.")

X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid, test_size=0.25,
    stratify=y_train_valid, random_state=42
)
y_train.value_counts(), y_valid.value_counts(), y_test.value_counts()

## 2. Baseline und Modell

Zuerst bewerten wir einen Mehrheitsklassen-Klassifikator. Danach trainieren wir einen Random Forest. Der Median-Imputer wird in der Pipeline nur anhand der jeweiligen Trainingsdaten angepasst; dadurch gelangen keine Informationen aus Validierung oder Test in die Datenvorbereitung.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_valid)
print(classification_report(
    y_valid, baseline_predictions,
    target_names=["neg", "pos"], zero_division=0
))

In [ ]:
model = make_pipeline(
    SimpleImputer(strategy="median"),
    RandomForestClassifier(
        n_estimators=100,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        n_jobs=-1,
        random_state=42,
    ),
)
model.fit(X_train, y_train)
valid_scores = model.predict_proba(X_valid)[:, 1]
print("PR-AUC (Validierung):", average_precision_score(y_valid, valid_scores))
print("ROC-AUC (Validierung):", roc_auc_score(y_valid, valid_scores))

## 3. Schwellenwert anhand der Fehlerkosten waehlen

Ein niedrigerer Schwellenwert kann mehr positive Faelle erkennen, fuehrt aber meist auch zu mehr Fehlalarmen. Wir waehlen ihn auf dem Validierungsdatensatz mit der Kostenfunktion der UCI-Herausforderung. Die Testdaten bleiben bis zur abschliessenden Bewertung unberuehrt.

In [ ]:
# Auch kleine Schwellenwerte prüfen: Ein übersehener Fehler kostet deutlich mehr als ein Fehlalarm.
thresholds = np.arange(0.01, 1.00, 0.01)
validation_costs = []
for threshold in thresholds:
    predictions = (valid_scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, predictions, labels=[0, 1]).ravel()
    validation_costs.append(10 * fp + 500 * fn)

best_threshold = thresholds[int(np.argmin(validation_costs))]
best_threshold, min(validation_costs)

## 4. Einmalige Bewertung auf dem Testdatensatz

Nach der Wahl des Schwellenwerts trainieren wir die Pipeline erneut auf Training und Validierung. Die Testdaten dienen nur der abschliessenden Schaetzung.

In [ ]:
model.fit(X_train_valid, y_train_valid)
test_scores = model.predict_proba(X_test)[:, 1]
test_predictions = (test_scores >= best_threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(
    y_test, test_predictions, labels=[0, 1]
).ravel()
print(classification_report(
    y_test, test_predictions,
    target_names=["neg", "pos"], zero_division=0
))
print("Gewaehlter Schwellenwert:", best_threshold)
print("PR-AUC (Test):", average_precision_score(y_test, test_scores))
print("ROC-AUC (Test):", roc_auc_score(y_test, test_scores))
print("UCI-Kostenmetrik (Test):", 10 * fp + 500 * fn)
print("Fehlalarme FP:", fp, "| uebersehene Fehler FN:", fn)

## 5. Einordnung und Reflexion

- Vergleicht Recall und Precision der positiven Klasse mit der Mehrheitsklassen-Baseline.
- Erklaert, warum sich die Zahl der Fehlalarme und uebersehenen APS-Fehler mit dem Schwellenwert aendert.
- Diskutiert, ob die UCI-Kostenannahmen fuer eine konkrete Wartungsorganisation plausibel waeren.
- Benennt die Grenzen: anonymisierte Merkmale, fehlende Werte, keine zeitliche Abfolge und keine Aussage zur Ursache oder zum Zeitpunkt eines Fehlers.

Ein gutes Ergebnis auf diesem Benchmark belegt noch nicht, dass das Modell in einer realen Flotte sicher oder wirtschaftlich eingesetzt werden kann.